# Suite PYP — The Python Patterns dialect

The patterns of Python's `match` statement, a grammar of their own (the Language Reference, 8.6.4): the dialect
`Expressions.Python.Patterns` of the Python family, its kinds mbse-programs' patterns, its `Term` of them alone, the
expressions they hold Python3's, its holder, what a pattern binds, its editions, its printing, and its evaluation, which
refuses each kind by name (the dialects proposal, Retrofit's design, Python).

In [ ]:
import functools
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Patterns import Expressions as Patterns
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3 import Evaluators as PV, Expressions as P, Text as PText
from mbse.Expressions.Python.Patterns import Expressions as PP, Text as PPText
from mbse.Expressions.Python.RegExp import Expressions as PR
from mbse.Expressions.Python.Types import Expressions as PT
from mbse.Expressions.Framework import Domains as D, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators, YAML
from support import raises

store = Builders.Store()  # the framework, the standard dialects Python3's complex numbers use, and the Python family
for install in [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
                PythonTypes.install, PythonPatterns.install]:
    store.transaction(install)
store.transaction(PythonPatterns.define)  # the versions the Patterns dialect covers
Term = store.registered("Expressions.Framework.Term")
READER = {d.union: d for d in (P.DIALECT, PT.DIALECT, PP.DIALECT)}  # the dialect whose union holds a term


def held(spec, dialect=PP.DIALECT):
    """The holder of the term `spec` writes, `dialect`'s own (the Patterns dialect's unless given), written in a
    transaction."""
    with store.transaction():
        holders = functools.reduce(getattr, dialect.path.split("."), store.Expressions)
        return holders.Expression(lambda e: e.root(dialect.spec(spec)))


pattern = lambda spec: F.term(held(spec))  # noqa: E731: a term of the Patterns dialect, held by its holder
capture = lambda name: lambda u: u.matchas(lambda m: m.name(name))  # noqa: E731: a capture pattern, `name`
wildcard = lambda u: u.matchas(lambda m: m)  # noqa: E731: `_`
value = lambda spec: lambda u: u.matchvalue(lambda m: m.value(P.DIALECT.spec(spec)))  # noqa: E731: a value pattern, a Python3 term or a native
sequence = lambda *items: lambda u: u.matchsequence(lambda s: s.patterns(list(items)))  # noqa: E731: an open sequence
listed = lambda *items: lambda u: u.matchsequence(lambda s: s.delimiters("[]").patterns(list(items)))  # noqa: E731: `[...]`
either = lambda *items: lambda u: u.matchor(lambda o: o.patterns(list(items)))  # noqa: E731: `a | b`
named = lambda inner, name: lambda u: u.matchas(lambda m: m.pattern(inner).name(name))  # noqa: E731: `inner as name`

<a id="PYP-01"></a>
## PYP-01 · The Patterns dialect's union, its kinds' schemas in mbse-programs' shapes, the expressions they hold Python3's, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
# Its kinds: mbse-programs' patterns, named and shaped as it has them; its Term lists them and invalid syntax alone, an
# expression in a pattern standing in a kind's own slot, which holds Python3's Term; it has no helper of its own, its
# names str.
union = lambda dialect: [b.name for b in store.registered(dialect.union).branches]  # noqa: E731: its Term's branches
assert PP.TAGS == ("matchvalue", "matchsingleton", "matchsequence", "matchmapping", "matchclass", "matchstar", "matchas",
                   "matchor") and PP.HELPERS == ()
assert union(PP.DIALECT) == [*PP.TAGS, "error", "missing"] and list(PP.DIALECT.kinds()) == [*PP.TAGS, "error", "missing"]
assert not set(PP.TAGS) & (set(union(P.DIALECT)) | set(union(PT.DIALECT)))  # no other union of the family lists them
assert PP.DIALECT.name() == "Python Patterns" and PP.DIALECT.tag == "python_patterns"
assert PP.DIALECT.path == "Python.Patterns" and PP.DIALECT.union == "Expressions.Python.Patterns.Term"
PatternsTerm, Python3Term = store.registered(PP.DIALECT.union), store.registered(P.DIALECT.union)
kinds = {b.name: b.schema for b in PatternsTerm.branches}
assert [kinds[tag].name for tag in PP.TAGS] == [f"Expressions.Python.Patterns.{kind}" for kind in (
    "MatchValue", "MatchSingleton", "MatchSequence", "MatchMapping", "MatchClass", "MatchStar", "MatchAs", "MatchOr")]
assert {tag: list(kinds[tag].properties) for tag in PP.TAGS} == {  # each a value object, in mbse-programs' order
    "matchvalue": ["value"], "matchsingleton": ["value"], "matchsequence": ["delimiters", "patterns"],
    "matchmapping": ["keys", "patterns", "rest"], "matchclass": ["cls", "patterns", "kwd_attrs", "kwd_patterns"],
    "matchstar": ["name"], "matchas": ["pattern", "name"], "matchor": ["patterns"]} and not any(kinds[t].ref for t in PP.TAGS)
assert kinds["matchvalue"].properties["value"].schema is kinds["matchclass"].properties["cls"].schema is Python3Term
assert kinds["matchmapping"].properties["keys"].schema.element is Python3Term  # a key: a literal or a value pattern's name
assert kinds["matchas"].properties["pattern"].schema is PatternsTerm and all(
    kinds[tag].properties[p].schema.element is PatternsTerm for tag, p in [("matchsequence", "patterns"), (
        "matchmapping", "patterns"), ("matchclass", "patterns"), ("matchclass", "kwd_patterns"), ("matchor", "patterns")])
assert F._parts(kinds["matchmapping"]) == ((), ("keys", "patterns"), ("rest",))  # two containers of terms (API 210)
assert F._parts(kinds["matchclass"]) == (("cls",), ("patterns", "kwd_patterns"), ("kwd_attrs",))
assert F._parts(kinds["matchas"]) == (("pattern",), (), ("name",)) and F._parts(kinds["matchsingleton"]) == ((), (), ("value",))
# No position of Python3's or of the Types dialect's holds its Term: a match is a statement, mbse-programs' `case`
# holding its pattern through the Patterns dialect's holder. Its Term is a branch of the framework's, its dialect a
# singleton.
holds = lambda schema: any(PatternsTerm in (p.schema, getattr(p.schema, "element", None)) for p in schema.properties.values())  # noqa: E731
assert not any(holds(b.schema) for d in (P.DIALECT, PT.DIALECT) for b in store.registered(d.union).branches)
assert {b.name: b.schema for b in Term.branches}["python_patterns"] is PatternsTerm
assert store.singleton("Expressions.Python.Patterns") is store.Instances.Expressions.Python.Patterns
# Each dialect of the family reads its kinds, and it reads theirs.
assert P.DIALECT.kind_named("matchas") is PP.DIALECT.kinds()["matchas"] and PP.DIALECT.kind_named("binop") is P.DIALECT.kinds()["binop"]
assert PP.DIALECT.reads() == (P.DIALECT, PT.DIALECT, PR.DIALECT) and PT.DIALECT.reads() == (P.DIALECT, PP.DIALECT, PR.DIALECT)
# Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
# validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
Holder = store.registered("Expressions.Python.Patterns.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is PatternsTerm and Holder.validate() == [] and list(Holder.adjacencies) == ["abi"]
every = either(  # noqa: E731: Point(0, y=[a, *rest]) | {'k': None, **kw} | (Color.RED as c)
    lambda u: u.matchclass(lambda c: c.cls(P.name("Point")).patterns([value(0)]).kwd_attrs(["y"]).kwd_patterns([
        listed(capture("a"), lambda s: s.matchstar(lambda m: m.name("rest")))])),
    lambda u: u.matchmapping(lambda m: m.keys([P.constant("k")]).patterns([lambda s: s.matchsingleton(lambda v: v.value("None"))]).rest("kw")),
    named(value(P.attribute(P.name("Color"), "RED")), "c"))
with store.transaction():
    holder = store.Expressions.Python.Patterns.Expression(lambda e: e.root(PP.DIALECT.spec(every)))
    linked = store.Schemas.OfExpression(lambda s: s.name("pyp.every").expression(lambda a: a.insert(lambda e: e.expression(holder))))
assert Validators.Validate(store)(Holder, holder) == [] and PP.DIALECT.validate(holder, {"Point", "Color"}) == []
assert [F.same(entry.expression, holder) for entry in Builders.OfExpression.Builder(linked, False, store).expression] == [True]
SOURCE = "Point(0, y=[a, *rest]) | {'k': None, **kw} | (Color.RED as c)"
assert F.dialect_of(F.term(holder)) == PP.DIALECT.union and PPText.ToText(holder) == SOURCE
for write, read in [(JSON.ToJSON(store), JSON.FromJSON(store)), (YAML.ToYAML(store), YAML.FromYAML(store))]:
    assert F.same(read(Holder, write(Holder, holder)).root, holder.root)
    assert F.same(read(PatternsTerm, write(PatternsTerm, F.term(holder))), F.term(holder))
# The schema refuses a pattern where a union that lacks it is held: Python3's holder's root.
with raises(Exception, match="'matchas' is not a member of a union value"):
    with store.transaction():
        store.Expressions.Python.Python3.Expression(lambda e: e.root(PP.DIALECT.make(F.Form("matchas", {}, (None,)))))
# Another family's term is refused, as before.
with raises(ValueError, match="a Patterns let is not a Python Patterns term"):
    PP.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1, Patterns.Expressions.variable("a")), Patterns.Expressions.DIALECT)))

<a id="PYP-02"></a>
## PYP-02 · Each kind and its checks: mbse-programs' (a choice among a few, a pattern for each key and keyword, two alternatives) and `ast`'s, its capture names; forms of two containers and `make`, and inference

In [ ]:
# Each kind's checks: mbse-programs' (a singleton and delimiters among a few, a mapping's pattern for each key, a
# class's pattern for each keyword, an `or` of two patterns), `ast`'s (an `as` pattern with a pattern has a name), and
# its names' (an identifier; a capture name not `_`, which captures nothing). A value pattern's value, a mapping's keys
# and a class's class are Python3's terms, their names references the scope must bind.
validate = lambda spec, bound=(): PP.DIALECT.validate(pattern(spec), bound)  # noqa: E731
singleton = lambda text: lambda u: u.matchsingleton(lambda v: v.value(text))  # noqa: E731
mapping = lambda keys, patterns, rest=None: lambda u: u.matchmapping(  # noqa: E731
    lambda m: (m.keys(keys).patterns(patterns) if rest is None else m.keys(keys).patterns(patterns).rest(rest)))
for spec, problems in [
    (value(1), []), (value(P.unaryop("-", 1)), []), (value(P.attribute(P.name("Color"), "RED")), ["value: value: variable 'Color' is not bound"]),
    (lambda u: u.matchvalue(lambda m: m), ["a matchvalue needs a value"]),
    (singleton("None"), []), (singleton("True"), []), (singleton("False"), []),
    (singleton("Maybe"), ["MatchSingleton.value cannot be 'Maybe'"]),
    (lambda u: u.matchsingleton(lambda v: v), ["a matchsingleton needs a value"]),
    (sequence(), []), (listed(capture("a"), wildcard), []),
    (lambda u: u.matchsequence(lambda s: s.delimiters("()").patterns([capture("a")])), []),
    (lambda u: u.matchsequence(lambda s: s.delimiters("{}")), ["MatchSequence.delimiters cannot be '{}'"]),
    (sequence(capture("1a")), ["argument 0: a matchas's name must be an identifier, got '1a'"]),
    (mapping([P.constant("k")], [capture("v")], "rest"), []), (mapping([], [], "rest"), []),
    (mapping([P.constant("k"), P.constant("j")], [capture("v")]), ["a MatchMapping needs a pattern for each key"]),
    (mapping([P.constant("k")], [capture("v")], "1r"), ["a matchmapping's rest must be an identifier, got '1r'"]),
    (mapping([P.constant("k")], [capture("v")], "_"), ["a matchmapping's rest cannot be '_'"]),
    (mapping([P.name("k")], [capture("1v")]), ["keys 0: variable 'k' is not bound", "patterns 0: a matchas's name must be an identifier, got '1v'"]),
    (lambda u: u.matchclass(lambda c: c.cls(P.name("int")).patterns([capture("n")])), []),  # `int(n)`
    (lambda u: u.matchclass(lambda c: c.cls(P.name("int")).kwd_attrs(["real", "imag"]).kwd_patterns([wildcard, value(0)])), []),
    (lambda u: u.matchclass(lambda c: c.patterns([wildcard])), ["a matchclass needs a cls"]),
    (lambda u: u.matchclass(lambda c: c.cls(P.name("int")).kwd_attrs(["real"])), ["a MatchClass needs a pattern for each keyword"]),
    (lambda u: u.matchclass(lambda c: c.cls(P.name("int")).kwd_attrs(["re al"]).kwd_patterns([wildcard])),
     ["a matchclass's kwd_attr must be an identifier, got 're al'"]),
    (lambda u: u.matchclass(lambda c: c.cls(P.name("int")).kwd_patterns([wildcard])), ["a MatchClass needs a pattern for each keyword"]),
    (lambda u: u.matchstar(lambda s: s.name("rest")), []), (lambda u: u.matchstar(lambda s: s), []),  # `*_`
    (lambda u: u.matchstar(lambda s: s.name("_")), ["a matchstar's name cannot be '_'"]),
    (lambda u: u.matchstar(lambda s: s.name("a b")), ["a matchstar's name must be an identifier, got 'a b'"]),
    (wildcard, []), (capture("x"), []), (named(listed(), "x"), []),
    (lambda u: u.matchas(lambda m: m.pattern(capture("x"))), ["a matchas with a pattern needs a name"]),
    (capture("_"), ["a matchas's name cannot be '_'"]),
    (named(capture("1x"), "y"), ["pattern: a matchas's name must be an identifier, got '1x'"]),
    (either(value(1), value(2)), []), (either(value(1)), ["a MatchOr needs two patterns"]),
]:
    assert validate(spec) == problems, (problems, validate(spec))
assert validate(value(P.attribute(P.name("Color"), "RED")), {"Color"}) == []
assert PP.problems(F.node("matchas", F.kind_of(pattern(wildcard))[1])) == []
# Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
# from which make writes it again, by the dialect whose union holds it; a mapping's and a class's two containers of
# terms are each given their size by a part named by the container (API 210), and read by container.
assert all(PP.DIALECT.kinds()[tag].role == F.APPLICATION for tag in PP.TAGS)
tree = pattern(every)
walked = list(F.walk(tree))
assert {F.kind_of(node)[0] for node in walked if F.dialect_of(node) == PP.DIALECT.union} == set(PP.TAGS)
for node in walked:
    dialect = READER[F.dialect_of(node)]
    made = F.term(held(dialect.make(F.form(node)), dialect))
    assert F.same(made, node) and made is not node
pairs = pattern(mapping([P.constant("k"), P.constant("j")], [capture("v"), wildcard], "kw"))
assert F.form(pairs).attributes == {"rest": "kw", "keys": F.Part("keys", {}, 2), "patterns": F.Part("patterns", {}, 2)}
node = F.node("matchmapping", F.kind_of(pairs)[1])
assert list(node.lists) == ["keys", "patterns"] and [len(held_) for held_ in node.lists.values()] == [2, 2] and len(node.many) == 4
point = pattern(lambda u: u.matchclass(lambda c: c.cls(P.name("Point")).patterns([capture("x")]).kwd_attrs(["y", "z"]).kwd_patterns([wildcard, value(0)])))
formed = F.form(point)
assert formed.attributes["kwd_attrs"].values == ("y", "z") and len(formed.arguments) == 4  # its class, then its containers
assert (formed.attributes["patterns"].size, formed.attributes["kwd_patterns"].size) == (1, 2)
assert F.form(pattern(wildcard)) == F.Form("matchas", {}, (None,)) and F.name_of(pattern(capture("x"))) == "x"
assert PP.DIALECT.infer(tree, {"Point": D.Anything, "Color": D.Anything}) is D.Anything

<a id="PYP-03"></a>
## PYP-03 · What a pattern binds: its capture names, wherever they nest, which the `case` holding it binds; none of its kinds a binder

In [ ]:
# What a pattern binds (`bound_names`): its capture names, an `as` pattern's and a star pattern's name and a mapping's
# rest, wherever its sequence, mapping, class, `as` and `or` patterns nest them; a value or a singleton pattern, the
# wildcard, `*_`, an expression a pattern holds (whose names are references) and invalid syntax bind none.
binds = lambda spec: PP.bound_names(pattern(spec))  # noqa: E731
assert binds(capture("a")) == {"a"} and binds(wildcard) == frozenset() and binds(value(P.name("a"))) == frozenset()
assert binds(singleton("None")) == frozenset() and binds(lambda u: u.matchstar(lambda s: s)) == frozenset()
assert binds(named(listed(capture("a"), lambda s: s.matchstar(lambda m: m.name("b"))), "c")) == {"a", "b", "c"}
assert binds(mapping([P.name("k")], [capture("v")], "rest")) == {"v", "rest"}  # not its key's name, a reference
assert binds(lambda u: u.matchclass(lambda c: c.cls(P.name("C")).patterns([capture("a")]).kwd_attrs(["x"]).kwd_patterns(
    [capture("b")]))) == {"a", "b"}  # not its class's name, nor its keyword's
assert binds(either(capture("a"), named(value(1), "a"))) == {"a"}
assert binds(sequence(lambda u: u.error(lambda e: e.text("a +").terms([PP.DIALECT.spec(capture("a"))])))) == frozenset()
# A pattern binds, but is no binder of its own: none of its kinds refines Let, and none has scopes; the `case` that holds
# it, mbse-programs' statement, binds its names in its guard and its body. Its expressions' names are free.
assert not any(F.is_binder(kinds[tag]) or PP.DIALECT.kinds()[tag].scopes is not None for tag in PP.TAGS)
assert SY.free(pattern(every), PP.DIALECT) == {"Point", "Color"}
assert SY.free(pattern(mapping([P.name("k")], [capture("v")])), PP.DIALECT) == {"k"}

<a id="PYP-04"></a>
## PYP-04 · A tree mixing patterns with Python3's expressions, validated, inferred and printed from the Patterns dialect's holder, and read by each dialect of the family

In [ ]:
# A tree mixing patterns with Python3's expressions: a value pattern's dotted name and literals (a negative number, a
# complex one), a mapping's keys, a class's class, each Python3's term in the pattern's own slot; validated, inferred and
# printed from the Patterns dialect's holder; every dialect of the family reads it.
mixed = either(  # noqa: E731
    lambda u: u.matchclass(lambda c: c.cls(P.attribute(P.name("shapes"), "Point")).kwd_attrs(["x", "y"]).kwd_patterns([
        value(P.unaryop("-", 1)), named(either(value(0), value(P.constant(2j))), "y")])),
    lambda u: u.matchmapping(lambda m: m.keys([P.constant("kind"), P.attribute(P.name("Keys"), "SIZE")]).patterns([
        value("circle"), sequence(capture("w"), capture("h"))])))
SOURCE = "shapes.Point(x=-1, y=0 | 2j as y) | {'kind': 'circle', Keys.SIZE: (w, h)}"
own = pattern(mixed)
assert PPText.ToText(own) == PText.ToText(own) == SOURCE and PPText.ToText is PText.ToText
assert PP.DIALECT.validate(own) == ["argument 0: cls: value: variable 'shapes' is not bound",
                                   "argument 1: keys 1: value: variable 'Keys' is not bound"]
assert all(dialect.validate(own, {"shapes", "Keys"}) == [] for dialect in (PP.DIALECT, P.DIALECT, PT.DIALECT))
assert SY.free(own, PP.DIALECT) == {"shapes", "Keys"} and PP.bound_names(own) == {"y", "w", "h"}
assert PP.DIALECT.infer(own, {"shapes": D.Anything, "Keys": D.Anything}) is D.Anything
cls = F.form(F.form(own).arguments[0]).arguments[0]  # the class pattern's class: Python3's term
assert F.dialect_of(cls) == P.DIALECT.union and F.dialect_of(F.form(own).arguments[0]) == PP.DIALECT.union

<a id="PYP-05"></a>
## PYP-05 · Invalid syntax where a pattern stands: validated, printed as written, and refused by evaluation

In [ ]:
# Invalid syntax: the Patterns dialect's Term holds `error` and `missing`, where a pattern stands; validation reports
# them, printers write them as written, and evaluation refuses them by the Patterns dialect's name.
broken = pattern(listed(lambda i: i.error(lambda e: e.text("a +").terms([PP.DIALECT.spec(capture("a"))])),
                        lambda i: i.missing(lambda m: m.kind("pattern"))))
assert PP.DIALECT.validate(broken) == ["argument 0: invalid syntax 'a +'", "argument 1: missing pattern"]
assert PText.ToText(broken) == "[a +, ]" and PP.bound_names(broken) == frozenset()
assert [PP.DIALECT.kind(a)[2].role for a in F.form(broken).arguments] == [F.INVALID, F.INVALID]
as_missing = pattern(lambda u: u.matchas(lambda m: m.pattern(lambda p: p.missing(lambda r: r)).name("x")))
assert PP.DIALECT.validate(as_missing) == ["pattern: missing a term"] and PText.ToText(as_missing) == " as x"
with raises(ValueError, match="Python Patterns error has no text to print"):
    PText.ToText(pattern(either(lambda e: e.error(lambda r: r), wildcard)))
with raises(ValueError, match="Python Patterns missing is not evaluated"):
    PV.OfAny(pattern(lambda u: u.missing(lambda m: m)))

<a id="PYP-06"></a>
## PYP-06 · Its editions as data: Python 3.10 to 3.15, each with every one of its own kinds

In [ ]:
# Its editions, as data: its define covers Python 3.10 to 3.15, owned by their published names, each with every one of
# its own kinds (all of them since Python 3.10, as mbse-programs' SINCE has them) and none of the kinds it reads; run
# again, it writes nothing.
patterns_ = store.Instances.Expressions.Python.Patterns
VERSIONS = tuple(f"Python 3.{minor}" for minor in range(10, 16))
assert tuple(FS.standards(patterns_)) == VERSIONS and PP.SINCE == dict.fromkeys(PP.TAGS, 10)
assert FS.tags(FS.find(patterns_, "Python 3.10")) == ("Python 3.10",) and FS.find(patterns_, "Python 3.9") is None
assert FS.properties(FS.find(patterns_, "Python 3.15")) == {"bodies": ("org.python",), "revision": ("version", "3.15")}
assert all(FS.kinds(edition) == [kinds[tag] for tag in PP.TAGS] for edition in FS.standards(patterns_).values())
assert all(tuple(FS.of(patterns_, kinds[tag])) == VERSIONS for tag in PP.TAGS)
assert FS.of(patterns_, Python3Term.branches[0].schema) == {} and FS.of(patterns_, store.registered("Expressions.Framework.Error")) == {}
assert store.transaction(PythonPatterns.define)[0]["Python_3_10"] is FS.find(patterns_, "Python 3.10")
assert tuple(FS.standards(patterns_)) == VERSIONS  # nothing written again

<a id="PYP-07"></a>
## PYP-07 · Printing every kind as mbse-programs writes patterns, parenthesized only where a pattern needs it

In [ ]:
# Printing, as mbse-programs' printer writes patterns (`p`): a value pattern as the expression it is, a singleton, a
# sequence in its delimiters or open (as a `case` writes it, at the root), a mapping, a class, a star, an `as` pattern,
# a capture and the wildcard, and an `or` pattern; an open sequence parenthesized where a pattern nests it, an `as`
# pattern where an `as` or an `or` pattern holds it, and an `or` pattern where an `or` pattern holds it.
cases = [
    (value(1), "1"), (value(P.unaryop("-", 1)), "-1"), (value("s"), "'s'"), (value(P.attribute(P.name("Color"), "RED")), "Color.RED"),
    (singleton("None"), "None"), (singleton("True"), "True"),
    (sequence(capture("a"), capture("b")), "a, b"), (sequence(capture("a")), "a,"), (sequence(), "()"),
    (listed(capture("a"), lambda s: s.matchstar(lambda m: m.name("rest"))), "[a, *rest]"), (listed(), "[]"),
    (lambda u: u.matchsequence(lambda s: s.delimiters("()").patterns([capture("a")])), "(a,)"),
    (lambda u: u.matchsequence(lambda s: s.delimiters("()").patterns([capture("a"), capture("b")])), "(a, b)"),
    (listed(capture("a"), sequence(capture("b"), capture("c"))), "[a, (b, c)]"),
    (listed(named(capture("a"), "b"), either(value(1), value(2))), "[a as b, 1 | 2]"),
    (mapping([], []), "{}"), (mapping([P.constant("k"), P.constant(1)], [capture("v"), wildcard], "rest"), "{'k': v, 1: _, **rest}"),
    (lambda u: u.matchclass(lambda c: c.cls(P.name("C"))), "C()"),
    (lambda u: u.matchclass(lambda c: c.cls(P.attribute(P.name("m"), "C")).patterns([capture("a"), sequence(capture("b"), capture("c"))])
     .kwd_attrs(["x"]).kwd_patterns([either(value(1), value(2))])), "m.C(a, (b, c), x=1 | 2)"),
    (lambda u: u.matchstar(lambda s: s.name("rest")), "*rest"), (lambda u: u.matchstar(lambda s: s), "*_"),
    (wildcard, "_"), (capture("x"), "x"), (named(listed(capture("a")), "x"), "[a] as x"),
    (named(named(capture("a"), "b"), "c"), "(a as b) as c"), (named(either(value(1), value(2)), "c"), "1 | 2 as c"),
    (named(sequence(capture("a"), capture("b")), "c"), "(a, b) as c"),
    (either(capture("a"), wildcard), "a | _"), (either(either(value(1), value(2)), value(3)), "(1 | 2) | 3"),
    (either(named(capture("a"), "b"), sequence(capture("c"), capture("d"))), "(a as b) | (c, d)"),
]
for spec, source in cases:
    assert PText.ToText(pattern(spec)) == source, (source, PText.ToText(pattern(spec)))
assert {F.kind_of(pattern(spec))[0] for spec, _ in cases} >= set(PP.TAGS)  # every kind
assert PPText.TEXTS.keys() == set(PP.TAGS) and PPText.OPEN < PPText.AS < PPText.OR < PPText.CLOSED

<a id="PYP-08"></a>
## PYP-08 · Evaluation refuses each of its kinds by the Patterns dialect's name

In [ ]:
# Evaluation (Q11): Python3's evaluator evaluates none of its kinds, and refuses each by the Patterns dialect's name; a
# pattern is matched by mbse-programs' `match` statement, which is not an expression.
for spec, tag in [(value(1), "matchvalue"), (singleton("None"), "matchsingleton"), (sequence(), "matchsequence"),
                  (mapping([], []), "matchmapping"), (lambda u: u.matchclass(lambda c: c.cls(P.name("C"))), "matchclass"),
                  (lambda u: u.matchstar(lambda s: s), "matchstar"), (wildcard, "matchas"), (either(wildcard, wildcard), "matchor")]:
    with raises(NotImplementedError, match=f"Python Patterns {tag} is not evaluated"):
        PV.OfAny(pattern(spec))

<a id="PYP-09"></a>
## PYP-09 · Completeness: each of `ast`'s pattern nodes, kept as data, is one of its kinds, and each of its kinds one of them; a `case` is a statement's part

In [ ]:
# Completeness (Q9): `grammars/python3-ast.json` lists `ast`'s pattern nodes (class `pattern`, Python 3.10 on), each
# mapped to one of the Patterns dialect's kinds, and every kind is one of them; `match_case`, a statement's part, is
# listed as not an expression (DIA-20 checks the file whole, and against the running Python's `ast`).
grammar = json.loads(Path("../../grammars/python3-ast.json").read_text(encoding="utf-8"))
entries = [node for node in grammar["nodes"] if node["class"] == "pattern"]
assert [(node["node"], node["since"], node["kinds"]) for node in entries] == [
    (kind, "3.10", [kind.lower()]) for kind in ("MatchValue", "MatchSingleton", "MatchSequence", "MatchMapping", "MatchClass",
                                                "MatchStar", "MatchAs", "MatchOr")]
assert {kind for node in entries for kind in node["kinds"]} == set(PP.TAGS)
case = next(node for node in grammar["nodes"] if node["node"] == "match_case")
assert case["class"] == "helper" and case["since"] == "3.10" and "not" in case